# POSSM pooled cache preparation

One-time Colab workflow that creates versioned Brain2Text25 area-6v caches. Brain2Text24 remains untouched in the canonical raw and smoothed roots. Raw and pre-smoothed Brain2Text25 are repacked independently so smoothing is never regenerated across new shard boundaries. Retained TX, SBP, and auxiliary arrays preserve their source dtypes and values exactly.


In [ ]:
# Mount Drive and edit this configuration cell only.

from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
DRIVE_DATA_ROOT = Path('/content/drive/MyDrive/utah_ssl/data')
RAW_SOURCE_ROOT = DRIVE_DATA_ROOT / 'cache_v1'
SMOOTHED_SOURCE_ROOT = DRIVE_DATA_ROOT / 'cache_v1_smoothed_sigma2p0'
RAW_DESTINATION_ROOT = DRIVE_DATA_ROOT / 'cache_v1_possm_b2t25_area6v_v1'
SMOOTHED_DESTINATION_ROOT = DRIVE_DATA_ROOT / 'cache_v1_possm_b2t25_area6v_sigma2p0_v1'
TARGET_SHARD_MB = 65.0

RUN_CACHE_BUILD = False  # Review the dry run, then set True.
RESUME_COMPLETED = True
REPLACE_PARTIAL = False
FORCE_RECOMPUTE_STATS = False
RUN_SAMPLING_BENCHMARK = True
BENCHMARK_BATCHES = 240

print({
    'raw_source': str(RAW_SOURCE_ROOT),
    'smoothed_source': str(SMOOTHED_SOURCE_ROOT),
    'raw_destination': str(RAW_DESTINATION_ROOT),
    'smoothed_destination': str(SMOOTHED_DESTINATION_ROOT),
    'target_shard_mb': TARGET_SHARD_MB,
})


In [ ]:
# Clone/update the repository and expose the experiment packages.

import os
import subprocess
import sys

REPO_URL = 'https://github.com/ethan-read/utah-ssl.git'
REPO_DIR = Path('/content/utah-ssl')
EXPERIMENTS_DIR = REPO_DIR / 'analysis' / 'active' / 'ssl_experiments'
POSSM_DIR = REPO_DIR / 'analysis' / 'reference' / 'possm'

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    status = subprocess.run(
        ['git', '-C', str(REPO_DIR), 'status', '--porcelain'],
        check=True, capture_output=True, text=True,
    )
    if not status.stdout.strip():
        subprocess.run(
            ['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', 'main'],
            check=True,
        )
    else:
        print('Using checkout with local changes; auto-pull skipped.')

for path in (REPO_DIR, EXPERIMENTS_DIR, POSSM_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
os.chdir(REPO_DIR)
print('repo:', REPO_DIR)


## 1. Dry-run inventory

This reads manifests and file metadata only. It does not create or alter cache roots.


In [ ]:
import json

PREP_SCRIPT = EXPERIMENTS_DIR / 'ssl_core/scripts/prepare_possm_pooled_cache.py'
BASE_PREP_COMMAND = [
    sys.executable, str(PREP_SCRIPT),
    '--raw-source-root', str(RAW_SOURCE_ROOT),
    '--smoothed-source-root', str(SMOOTHED_SOURCE_ROOT),
    '--raw-destination-root', str(RAW_DESTINATION_ROOT),
    '--smoothed-destination-root', str(SMOOTHED_DESTINATION_ROOT),
    '--target-mb', str(TARGET_SHARD_MB),
]
dry_run = subprocess.run(
    [*BASE_PREP_COMMAND, '--dry-run'],
    cwd=str(REPO_DIR), check=True, text=True, capture_output=True,
)
dry_payload = json.loads(dry_run.stdout)
print({key: dry_payload[key] for key in ('datasets', 'area6v_columns', 'tx_storage_policy', 'sbp_storage_policy', 'target_mb')})
for variant in ('raw', 'smoothed'):
    entry = dry_payload[variant]
    files = entry['source_inventory']['files']
    print({
        'variant': variant,
        'source': entry['source'],
        'destination': entry['destination'],
        'source_signature': entry['source_inventory']['source_signature'],
        'file_count': len(files),
        'total_gb': sum(int(row['size']) for row in files) / (1024 ** 3),
    })


## 2. Build and fully validate

After reviewing the inventory, set `RUN_CACHE_BUILD=True` in the configuration cell. Each root is written to a `.partial` sibling, validated example by example, and renamed only after success.


In [ ]:
if not RUN_CACHE_BUILD:
    print('Build skipped. Set RUN_CACHE_BUILD=True after reviewing the dry run.')
else:
    build_command = list(BASE_PREP_COMMAND)
    if RESUME_COMPLETED:
        build_command.append('--resume-completed')
    if REPLACE_PARTIAL:
        build_command.append('--replace-partial')
    print('Running:', ' '.join(build_command))
    subprocess.run(build_command, cwd=str(REPO_DIR), check=True)

for root in (RAW_DESTINATION_ROOT, SMOOTHED_DESTINATION_ROOT):
    summary_path = root / 'possm_pooled_cache_prep_summary.json'
    print(root, 'complete=', summary_path.exists())
    if summary_path.exists():
        payload = json.loads(summary_path.read_text())
        print(json.dumps(payload['validation']['datasets'], indent=2))


## 3. Recompute pooled session statistics

The pooled view combines canonical Brain2Text24 with optimized Brain2Text25 and therefore receives a separate composite-signature statistics artifact.


In [ ]:
if not SMOOTHED_DESTINATION_ROOT.is_dir():
    raise FileNotFoundError('Build the optimized smoothed cache before computing stats.')

STATS_VARIANT = 'possm_b2t24_canonical_b2t25_area6v_sigma2p0_v1'
STATS_PATH = (
    DRIVE_DATA_ROOT / 'stats/session_feature_stats' / STATS_VARIANT
    / 'tx_only/session/ssl_pretrain_b2t24_competition_train_b2t25_train_val_v1.pt'
)
stats_command = [
    sys.executable,
    str(EXPERIMENTS_DIR / 'ssl_core/scripts/recompute_session_feature_stats.py'),
    '--cache-root', str(SMOOTHED_SOURCE_ROOT),
    '--dataset-cache-root', f'brain2text25={SMOOTHED_DESTINATION_ROOT}',
    '--output-path', str(STATS_PATH),
    '--feature-mode', 'tx_only',
    '--boundary-key-mode', 'session',
    '--tx-dim', '128', '--sbp-dim', '128',
    '--segment-bins', '100', '--examples-per-shard', '8',
    '--dataset', 'brain2text24', '--dataset', 'brain2text25',
    '--dataset-source-split', 'brain2text24=competition_train',
    '--dataset-source-split', 'brain2text25=train',
    '--dataset-source-split', 'brain2text25=val',
]
stats_sidecar = STATS_PATH.with_suffix('.json')
from masked_ssl.cache import FEATURE_POLICY, _compute_dataset_cache_source_signature
expected_stats_signature = _compute_dataset_cache_source_signature({
    'brain2text24': SMOOTHED_SOURCE_ROOT,
    'brain2text25': SMOOTHED_DESTINATION_ROOT,
})
stats_are_current = False
if STATS_PATH.exists() and stats_sidecar.exists():
    try:
        stats_metadata = json.loads(stats_sidecar.read_text())
        stats_are_current = (
            stats_metadata.get('source_cache_signature') == expected_stats_signature
            and stats_metadata.get('feature_policy') == FEATURE_POLICY
            and stats_metadata.get('feature_mode') == 'tx_only'
            and stats_metadata.get('boundary_key_mode') == 'session'
            and stats_metadata.get('full_dim') == 128
            and stats_metadata.get('dataset_names') == ['brain2text24', 'brain2text25']
            and stats_metadata.get('pretrain_source_splits_by_dataset') == {
                'brain2text24': ['competition_train'],
                'brain2text25': ['train', 'val'],
            }
        )
    except (OSError, json.JSONDecodeError):
        stats_are_current = False
if FORCE_RECOMPUTE_STATS or not stats_are_current:
    stats_command.append('--overwrite')
if stats_are_current and not FORCE_RECOMPUTE_STATS:
    print('Reusing:', STATS_PATH)
else:
    subprocess.run(stats_command, cwd=str(REPO_DIR), check=True)
print('stats:', STATS_PATH)


In [ ]:
# Compare against the old pooled artifact when it is available.

import torch

OLD_STATS_PATH = (
    DRIVE_DATA_ROOT / 'stats/session_feature_stats/smoothed_sigma2p0/tx_only/session'
    / 'ssl_pretrain_b2t24_competition_train_b2t25_train_val_v1.pt'
)
if OLD_STATS_PATH.exists():
    old_payload = torch.load(OLD_STATS_PATH, map_location='cpu', weights_only=False)
    new_payload = torch.load(STATS_PATH, map_location='cpu', weights_only=False)
    old_stats = old_payload['session_feature_stats']
    new_stats = new_payload['session_feature_stats']
    assert set(old_stats) == set(new_stats)
    max_mean_delta = 0.0
    max_std_delta = 0.0
    for key in old_stats:
        max_mean_delta = max(max_mean_delta, float((old_stats[key][0] - new_stats[key][0]).abs().max()))
        max_std_delta = max(max_std_delta, float((old_stats[key][1] - new_stats[key][1]).abs().max()))
    # Repacking is lossless; allow only floating-point accumulation-order noise.
    assert max_mean_delta <= 1e-6 and max_std_delta <= 1e-6
    print({'max_mean_delta': max_mean_delta, 'max_std_delta': max_std_delta})
else:
    print('Old pooled stats artifact not found; logical cache validation remains authoritative.')


## 4. Cold-to-warm sampling benchmark

This creates a separate sampler and does not advance the sampler used by training.


In [ ]:
if RUN_SAMPLING_BENCHMARK:
    import time
    import numpy as np
    from masked_ssl.cache import CacheAccessConfig, prepare_cache_context
    from possm_ssl.training import build_possm_segment_sampler

    split_policy = {
        'brain2text24': ('competition_train',),
        'brain2text25': ('train', 'val'),
    }
    benchmark_context = prepare_cache_context(
        cache_candidates=[SMOOTHED_SOURCE_ROOT],
        config=CacheAccessConfig(
            mode='drive_direct', included_datasets=tuple(split_policy), seed=1707,
            segment_bins=100, use_normalization=True, examples_per_shard=8,
            tx_dim=128, sbp_dim=128, feature_mode='tx_only',
            boundary_key_mode='session', precomputed_session_stats_path=STATS_PATH,
            pretrain_source_splits_by_dataset=split_policy,
            dataset_cache_roots={'brain2text25': SMOOTHED_DESTINATION_ROOT},
        ),
    )
    benchmark_sampler = build_possm_segment_sampler(
        cache_context=benchmark_context, split_name='train', batch_size=32,
        seed=1707, segment_bins=100, dataset_weight_alpha=0.25,
        examples_per_shard=8, data_mode='normalized',
    )
    timings = []
    for _ in range(int(BENCHMARK_BATCHES)):
        t0 = time.perf_counter()
        benchmark_sampler.sample_batch()
        timings.append(time.perf_counter() - t0)
    window = max(20, int(BENCHMARK_BATCHES) // 4)
    cold = np.asarray(timings[:window])
    warm = np.asarray(timings[-window:])
    report = {
        'cold_mean_s': float(cold.mean()),
        'warm_mean_s': float(warm.mean()),
        'warm_median_s': float(np.median(warm)),
        'warm_p90_s': float(np.quantile(warm, 0.90)),
        'speedup_cold_to_warm': float(cold.mean() / max(warm.mean(), 1e-9)),
        'shard_cache': benchmark_context.shard_store.summary(),
    }
    print(json.dumps(report, indent=2))
    if report['warm_mean_s'] > 0.10:
        print('Warning: warm sampling remains above 0.10 s; try copy_to_local in s14.')
else:
    print('Sampling benchmark skipped.')
